In [ ]:
!pip install datasets pandas
!pip install -q groq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import json
import os
import gzip
from datasets import load_dataset
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks'

config = {
    "mgsm_en": os.path.join(base_path, "mgsm_en.tsv"),
    "mgsm_zh": os.path.join(base_path, "mgsm_zh.tsv"),
    "mgsm_es": os.path.join(base_path, "mgsm_es.tsv"),
    "mkqa_local": os.path.join(base_path, "mkqa.jsonl.gz"),
    "num_samples_per_type": 150
}

def build_final_dataset():
    final_dataset = []

    # --- 2. MGSM  ---
    print("getting MGSM...")
    try:
        df_en = pd.read_csv(config['mgsm_en'], sep='\t', header=None)
        df_zh = pd.read_csv(config['mgsm_zh'], sep='\t', header=None)
        df_es = pd.read_csv(config['mgsm_es'], sep='\t', header=None)
        limit = min(len(df_en), config['num_samples_per_type'])
        for i in range(limit):
            final_dataset.append({
                "id": f"mgsm_{i:03d}",
                "type": "MGSM",
                "ground_truth": str(df_en.iloc[i, 1]),
                "languages": {
                    "en": {"question": df_en.iloc[i, 0], "prompt": f"Solve this math problem. Answer with only the final number: {df_en.iloc[i, 0]}", "model_output": [], "is_correct": None},
                    "zh": {"question": df_zh.iloc[i, 0], "prompt": f"解决这个数学问题。只回答最终数字：{df_zh.iloc[i, 0]}", "model_output": [], "is_correct": None},
                    "es": {"question": df_es.iloc[i, 0], "prompt": f"Resuelve este problema matemático. Responde solo con el número final: {df_es.iloc[i, 0]}", "model_output": [], "is_correct": None}
                }
            })
        print(f" MGSM complete：{limit} itms")
    except Exception as e: print(f" MGSM error (Check file name): {e}")

    # --- 3. XQuAD (Hugging Face) ---
    print("getting XQuAD...")
    try:
        x_en = load_dataset("google/xquad", "xquad.en", split="validation")
        x_zh = load_dataset("google/xquad", "xquad.zh", split="validation")
        x_es = load_dataset("google/xquad", "xquad.es", split="validation")
        limit = min(len(x_en), config['num_samples_per_type'])
        for i in range(limit):
            final_dataset.append({
                "id": f"xquad_{i:03d}",
                "type": "XQuAD",
                "context_en": x_en[i]['context'],
                "ground_truth": x_en[i]['answers']['text'][0] if x_en[i]['answers']['text'] else "",
                "languages": {
                    "en": {"question": x_en[i]['question'], "prompt": f"Answer concisely based on the context: {x_en[i]['question']}", "model_output": [], "is_correct": None},
                    "zh": {"question": x_zh[i]['question'], "prompt": f"请根据背景简洁回答：{x_zh[i]['question']}", "model_output": [], "is_correct": None},
                    "es": {"question": x_es[i]['question'], "prompt": f"Responde concisamente según el contexto: {x_es[i]['question']}", "model_output": [], "is_correct": None}
                }
            })
        print(f" XQuAD complete：{limit} items")
    except Exception as e: print(f" XQuAD error: {e}")

    # --- 4.  MKQA (JSONL.GZ) ---
    print(f"Reading from the local path MKQA: {config['mkqa_local']}...")
    try:
        count = 0

        with gzip.open(config['mkqa_local'], 'rt', encoding='utf-8') as f:
            for line in f:
                if count >= config['num_samples_per_type']: break
                item = json.loads(line)

                queries = item['queries']
                zh_key = 'zh_cn' if 'zh_cn' in queries else 'zh-cn'

                if 'en' in queries and zh_key in queries and 'es' in queries:

                    gt = ""
                    if 'en' in item.get('answers', {}):
                        ans_list = item['answers']['en']
                        if ans_list and 'text' in ans_list[0]:
                            gt = ans_list[0]['text']

                    final_dataset.append({
                        "id": f"mkqa_{count:03d}",
                        "type": "MKQA",
                        "ground_truth": gt,
                        "languages": {
                            "en": {"question": queries['en'], "prompt": f"Answer this factual question concisely: {queries['en']}", "model_output": [], "is_correct": None},
                            "zh": {"question": queries[zh_key], "prompt": f"请简洁回答这个事实性问题：{queries[zh_key]}", "model_output": [], "is_correct": None},
                            "es": {"question": queries['es'], "prompt": f"Responde concisamente a esta pregunta factual: {queries['es']}", "model_output": [], "is_correct": None}
                        }
                    })
                    count += 1
        print(f" MKQA complete：{count} items")
    except Exception as e: print(f" Error report (Check path: {e}")

    return final_dataset

full_data = build_final_dataset()
with open('g24_full_multilingual_data.json', 'w', encoding='utf-8') as f:
    json.dump(full_data, f, ensure_ascii=False, indent=2)

print(f"The full dataset has been generated：g24_full_multilingual_data.json")